# AI-Task6: E-Commerce Order Validator (Pydantic v2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task6/AI_Task6_Order_Validator.ipynb)

**Kya banana hai:** JSON orders ko Pydantic se validate karna, saari ghaltiyan jama karna (crash nahi), sahi orders `model_dump_json()` se save karna aur ghalat orders ki alag report banana.

**Concepts:** BaseModel, Field, ConfigDict, field_validator, model_validator, computed_field, field_serializer, model_dump_json, model_validate, RootModel

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

Is task ko API key ya internet nahi chahiye.

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Install
(Colab mein pydantic pehle se hota hai, phir bhi v2 pakka kar lete hain.)

In [1]:
!pip -q install -U pydantic

## 2. Models (`models.py`)
`%%writefile` is cell ko file bana deta hai, taake `from models import ...` chale.

- `Customer`: email ka `pattern`, naam Title Case (`@field_validator`)
- `Product`: price `gt=0`
- `OrderItem`: quantity `ge=1`, `line_total` (`@computed_field`)
- `Order`: status `Literal`, `mode="before"` validator, total check (`@model_validator`), `total_with_tax` (17% GST)

In [2]:
%%writefile models.py
from typing import Literal, Optional

from pydantic import (BaseModel, ConfigDict, Field, computed_field, field_serializer,
                      field_validator, model_validator)

TAX_RATE = 0.17          # 17% GST
EMAIL_PATTERN = r"^[\w.+-]+@[\w-]+\.[\w.]+$"
Status = Literal["Pending", "Paid", "Shipped", "Delivered", "Cancelled"]


class Customer(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)     # naam ke aage peeche ki khali jagah hata do

    name: str = Field(min_length=2, max_length=60)
    email: str = Field(pattern=EMAIL_PATTERN)
    city: Optional[str] = None                                # v2 style: Optional ki jagah  str | None  bhi chalta

    @field_validator("name", "city")
    @classmethod
    def title_case(cls, value):
        # saari strings Title Case mein (None ho to waise hi chhor do)
        return value.title() if value else value

    @field_validator("email")
    @classmethod
    def lower_email(cls, value):
        return value.lower()                                  # email ko title case nahi karte, lowercase rakhte hain


class Product(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    sku: str = Field(pattern=r"^[A-Z]{2}-\d{3}$")             # jaise  SH-101
    name: str = Field(min_length=2)
    price: float = Field(gt=0)                                # price zaroor positive

    @field_validator("name")
    @classmethod
    def title_case(cls, value):
        return value.title()


class OrderItem(BaseModel):
    product: Product
    quantity: int = Field(ge=1)                               # kam az kam 1

    @computed_field
    @property
    def line_total(self) -> float:
        return round(self.product.price * self.quantity, 2)


class Order(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    order_id: str = Field(pattern=r"^ORD-\d{4}$")
    customer: Customer
    items: list[OrderItem] = Field(min_length=1)
    status: Status = "Pending"
    total: float = Field(gt=0)

    @field_validator("status", mode="before")
    @classmethod
    def normalise_status(cls, value):
        # mode="before": Literal check se PEHLE chalta hai, is liye "PAID" / "paid" dono "Paid" ban jate hain
        return value.strip().title() if isinstance(value, str) else value

    @model_validator(mode="after")
    def total_matches_items(self):
        expected = round(sum(item.line_total for item in self.items), 2)
        if abs(expected - self.total) > 0.01:
            raise ValueError(f"total {self.total} line items ke jama {expected} se match nahi karta")
        return self

    @computed_field
    @property
    def total_with_tax(self) -> float:
        return round(self.total * (1 + TAX_RATE), 2)

    @field_serializer("total")
    def round_total(self, value):
        return round(value, 2)

Writing models.py


## 3. Raw orders (`orders.json`)
Paanch orders: 2 sahi, 3 jaan boojh kar ghalat (quantity 0, bura email, ghalat status, negative price, total match nahi karta).

In [3]:
%%writefile orders.json
[
  {
    "order_id": "ORD-1001",
    "customer": {"name": "  ayesha khan ", "email": "Ayesha.Khan@Example.com", "city": "narowal"},
    "items": [
      {"product": {"sku": "SH-101", "name": "cotton shirt", "price": 1500}, "quantity": 2},
      {"product": {"sku": "CP-202", "name": "baseball cap", "price": 500}, "quantity": 1}
    ],
    "status": "PAID",
    "total": 3500
  },
  {
    "order_id": "ORD-1002",
    "customer": {"name": "bilal ahmed", "email": "bilal@example.com"},
    "items": [{"product": {"sku": "SH-101", "name": "cotton shirt", "price": 1500}, "quantity": 0}],
    "status": "pending",
    "total": 0
  },
  {
    "order_id": "ORD-1003",
    "customer": {"name": "sana malik", "email": "sana-at-example.com"},
    "items": [{"product": {"sku": "BG-303", "name": "leather bag", "price": -2500}, "quantity": 1}],
    "status": "Packed",
    "total": 2500
  },
  {
    "order_id": "ORD-1004",
    "customer": {"name": "hamza ali", "email": "hamza@example.com", "city": "lahore"},
    "items": [
      {"product": {"sku": "SH-101", "name": "cotton shirt", "price": 1500}, "quantity": 1},
      {"product": {"sku": "CP-202", "name": "baseball cap", "price": 500}, "quantity": 2}
    ],
    "status": "shipped",
    "total": 9999
  },
  {
    "order_id": "ORD-1005",
    "customer": {"name": "maryam noor", "email": "maryam@example.com", "city": "sialkot"},
    "items": [{"product": {"sku": "BG-303", "name": "leather bag", "price": 2500}, "quantity": 1}],
    "status": "delivered",
    "total": 2500
  }
]

Writing orders.json


## 4. Validation ke functions

In [4]:
import json
import os

from pydantic import RootModel, ValidationError

from models import Order

In [5]:
class ValidatedOrders(RootModel[list[Order]]):
    """RootModel: poori list ko ek model bana diya, taake model_dump_json() seedha chale."""


def load_orders(path):
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        raise SystemExit(f"File nahi mili: {path}")
    except json.JSONDecodeError as e:
        raise SystemExit(f"JSON file kharab hai (line {e.lineno}): {e.msg}")
    if not isinstance(data, list):
        raise SystemExit("File mein orders ki list honi chahiye: [ {...}, {...} ]")
    return data


def validate_all(raw_orders):
    good, errors = [], []
    for index, raw in enumerate(raw_orders, start=1):
        order_id = raw.get("order_id", f"#{index}") if isinstance(raw, dict) else f"#{index}"
        try:
            good.append(Order.model_validate(raw))
        except ValidationError as e:
            # e.errors() har ghalti ki list deta hai: kahan (loc) aur kya (msg)
            for err in e.errors():
                where = ".".join(str(part) for part in err["loc"]) or "order"
                errors.append({"order_id": order_id, "field": where, "message": err["msg"]})
    return good, errors


def write_outputs(good, errors):
    os.makedirs("output", exist_ok=True)

    with open("output/validated_orders.json", "w", encoding="utf-8") as f:
        f.write(ValidatedOrders(good).model_dump_json(indent=2))

    with open("output/error_report.txt", "w", encoding="utf-8") as f:
        bad_orders = sorted({e["order_id"] for e in errors})
        f.write(f"ERROR REPORT: {len(errors)} ghaltiyan, {len(bad_orders)} ghalat orders\n")
        f.write("=" * 60 + "\n")
        for e in errors:
            f.write(f"{e['order_id']} | {e['field']} | {e['message']}\n")

    # model_json_schema(): model ka schema (LLM ko structure batane ke kaam aata hai)
    with open("output/order_schema.json", "w", encoding="utf-8") as f:
        json.dump(Order.model_json_schema(), f, indent=2)

## 5. Chalao

In [6]:
raw_orders = load_orders("orders.json")
good, errors = validate_all(raw_orders)
write_outputs(good, errors)

print(f"Kul orders : {len(raw_orders)}")
print(f"Sahi       : {len(good)}")
print(f"Ghalat     : {len(raw_orders) - len(good)}\n")
for order in good:
    print(f"OK   {order.order_id}  {order.customer.name:<14} {order.status:<10} "
          f"total={order.total:>8.2f}  with tax={order.total_with_tax:>8.2f}")
for e in errors:
    print(f"FAIL {e['order_id']}  {e['field']}: {e['message']}")

Kul orders : 5
Sahi       : 2
Ghalat     : 3

OK   ORD-1001  Ayesha Khan    Paid       total= 3500.00  with tax= 4095.00
OK   ORD-1005  Maryam Noor    Delivered  total= 2500.00  with tax= 2925.00
FAIL ORD-1002  items.0.quantity: Input should be greater than or equal to 1
FAIL ORD-1002  total: Input should be greater than 0
FAIL ORD-1003  customer.email: String should match pattern '^[\w.+-]+@[\w-]+\.[\w.]+$'
FAIL ORD-1003  items.0.product.price: Input should be greater than 0
FAIL ORD-1003  status: Input should be 'Pending', 'Paid', 'Shipped', 'Delivered' or 'Cancelled'
FAIL ORD-1004  order: Value error, total 9999.0 line items ke jama 2500.0 se match nahi karta


## 6. Bani hui files dekho

In [7]:
print(open("output/error_report.txt").read())
print(open("output/validated_orders.json").read()[:700])

ERROR REPORT: 6 ghaltiyan, 3 ghalat orders
ORD-1002 | items.0.quantity | Input should be greater than or equal to 1
ORD-1002 | total | Input should be greater than 0
ORD-1003 | customer.email | String should match pattern '^[\w.+-]+@[\w-]+\.[\w.]+$'
ORD-1003 | items.0.product.price | Input should be greater than 0
ORD-1003 | status | Input should be 'Pending', 'Paid', 'Shipped', 'Delivered' or 'Cancelled'
ORD-1004 | order | Value error, total 9999.0 line items ke jama 2500.0 se match nahi karta

[
  {
    "order_id": "ORD-1001",
    "customer": {
      "name": "Ayesha Khan",
      "email": "ayesha.khan@example.com",
      "city": "Narowal"
    },
    "items": [
      {
        "product": {
          "sku": "SH-101",
          "name": "Cotton Shirt",
          "price": 1500.0
        },
        "quantity": 2,
        "line_total": 3000.0
      },
      {
        "product": {
          "sku": "CP-202",
          "name": "Baseball Cap",
          "price": 500.0
        },
        "quantit

## 7. Pydantic ke baaki methods (concepts)
`model_dump`, `model_validate_json`, `model_copy`, `model_json_schema`, `RootModel`, `TypeAdapter`.

In [8]:
from pydantic import TypeAdapter

order = good[0]

# model_dump: object -> dict          model_validate_json: JSON string -> object
as_dict = order.model_dump()
again = Order.model_validate_json(order.model_dump_json())
print("round trip theek:", again.total == order.total)

# model_copy: copy banao, kuch fields badal kar (asal object nahi badalta)
shipped = order.model_copy(update={"status": "Shipped"})
print(order.status, "->", shipped.status)

# model_json_schema: model ka schema (LLM ko structure batane ke kaam aata hai)
print("schema ke fields:", list(Order.model_json_schema()["properties"]))

# TypeAdapter: bina class banaye kisi bhi type ko validate karo
print(TypeAdapter(list[int]).validate_python(["1", "2", 3]))

# RootModel: list ko hi model bana do
Marks = RootModel[list[int]]
print(Marks([90, 85]).model_dump_json())

round trip theek: True
Paid -> Shipped
schema ke fields: ['order_id', 'customer', 'items', 'status', 'total']
[1, 2, 3]
[90,85]


## Kya seekha
- Pydantic galat data ko runtime par pakadta hai aur theek type mein badal deta hai (`"10"` -> `10.0`).
- `Field(gt=0, ge=1, pattern=...)` constraints, `Literal` allowed values.
- `mode="before"` validator type check se pehle chalta hai, `"after"` baad mein.
- `@model_validator` do fields ka rishta check karta hai (total = jama).
- `e.errors()` se saari ghaltiyan list mein milti hain, is liye program crash nahi hota.